# 권현성의 경제금융용어 RAG 챗봇

- **핵심 개념:** PDF 로딩, 임베딩, Chroma 검색, 근거 기반 답변, Gradio UI
- **나의 수정:** 검색 문서 수와 질문 예제를 바꾸고 쉬운 정의·예시 중심 응답 방식으로 개인화
- API 키는 코드에 저장하지 않고 Colab `Secrets`의 `OPENAI_API_KEY`를 사용합니다.


In [ ]:
!pip install -q langchain-core langchain-community langchain-openai langchain-text-splitters chromadb tiktoken pypdf matplotlib gradio

In [ ]:
import os
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Colab 왼쪽의 Secrets에 OPENAI_API_KEY를 등록하세요.")

os.environ["OPENAI_API_KEY"] = api_key


> **데이터 살펴보기**

* 데이터는 한국은행에서 제공하는 '2020_경제금융용어 700선_게시.pdf'를 사용한다.

In [ ]:
import os
import re
import getpass
import matplotlib.pyplot as plt
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

https://www.bok.or.kr/portal/bbs/B0000249/view.do?nttId=235017&menuNo=200765  

위 링크에서 pdf 파일을 다운로드 받아서 업로드합니다.  

2020_경제금융용어 700선_게시.pdf

PDF를 로드하여 여러 개의 문서로 분할해주는  

```from langchain.document_loaders import PyPDFLoader```를 사용합니다.

In [ ]:
!wget -O "2020_경제금융용어 700선_게시.pdf" "https://raw.githubusercontent.com/chatgpt-kr/openai-api-tutorial/main/ch07/2020_%EA%B2%BD%EC%A0%9C%EA%B8%88%EC%9C%B5%EC%9A%A9%EC%96%B4%20700%EC%84%A0_%EA%B2%8C%EC%8B%9C.pdf"

## **Data Spec Check**

사용할 데이터를 점검하는 단계입니다.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("2020_경제금융용어 700선_게시.pdf")
texts = loader.load_and_split()

* load_and_split()과 관련된 공식 문서:

https://api.python.langchain.com/en/latest/document_loaders/langchain.document_loaders.pdf.PyPDFLoader.html  

In [ ]:
print('문서의 수 :', len(texts))

PDF 문서가 총 366개의 텍스트로 분할되었습니다. 그 중 임의로 15번 문서를 출력해보겠습니다.

Document(metadata={'source': '파일명', 'page': 기존 파일에서 몇 페이지였는지}, page_content=실제 내용)

In [ ]:
texts[15]

형식은 page_content에는 분할된 텍스트의 본문이 저장되어져 있고, source에는 해당 본문의 원본 파일의 이름이 저장되어져 있습니다.

In [ ]:
# 문서를 직접 접근하기 위해서는 .page_content를 사용.
texts[15].page_content

texts에서 page_content. 즉, 각 문서의 본문만 추출하여 documents라는 변수에 저장해봅시다.

In [ ]:
documents = [text.page_content for text in texts]
print(documents[15])
print('-' * 100)
print('15번 문서의 길이 :', len(documents[15]))

이제 문서의 최대 길이, 최소 길이, 평균 길이, 그리고 길이 분포를 시각화해봅시다.

In [ ]:
print('문서의 최대 길이 :',max(len(document) for document in documents))
print('문서의 최소 길이 :',min(len(document) for document in documents))
print('문서의 평균 길이 :',sum(map(len, documents))/len(documents))
plt.hist([len(review) for review in documents], bins=50)
plt.xlabel('length of samples')
plt.ylabel('number of samples')
plt.show()

In [ ]:
# 0번 문서는 머리말
texts[0].page_content

In [ ]:
texts[5].page_content

In [ ]:
# 12번 문서까지는 목차
texts[12].page_content

In [ ]:
# 13번 문서부터는 금융 용어 설명
texts[13].page_content

참고로 \n은 줄바꿈 문자입니다. 쉽게 설명하면 Enter에 해당하므로 크게 신경쓰지 않아도 됩니다. print()로 출력해보면 알 수 있습니다.

In [ ]:
print(texts[13].page_content)

앞에 몇 개의 문서를 출력해본 결과 0번 문서는 머리말, 12번 문서까지는 목차, 13번 문서부터 금융 용어를 설명하는 문서임을 확인하였습니다. 다시 말해 용어를 검색을 위해서는 0번 문서부터 12번 문서는 필요가 없을 것입니다. 기존의 0번 문서와 12번 문서를 제거해보겠습니다.

In [ ]:
texts = texts[13:]
print('줄어든 texts의 길이 :', len(texts))

현재의 0번 문서를 출력해보겠습니다. 전처리가 제대로 되었다면 이전의 13번 문서가 현재의 0번 문서가 되어야 합니다.

In [ ]:
print('첫번째 문서 출력 :', texts[0])

이번에는 마지막 데이터를 확인해봅시다. 해당 문서에 대한 맺음말에 해당되므로 해당 데이터도 불필요합니다.

In [ ]:
texts[-1]

뒤에서 두번째 데이터를 확인해봅시다.

In [ ]:
texts[-2]

마지막 데이터를 제거 후에 개수를 출력해봅시다.

In [ ]:
# 마지막 데이터를 제거
texts = texts[:-1]
print('마지막 데이터 제거 후 texts의 길이 :', len(texts))

이제 데이터의 개수가 1개 줄어들었습니다. 이제 마지막 데이터를 출력하면 앞서 출력되었던 뒤에서 두번째 데이터가 출력되어야 합니다.

In [ ]:
print('마지막 데이터 출력')
texts[-1]

정리해봅시다. PyPDFLoader가 PDF를 분할하여 다수의 문서로 만들었습니다.

1. 문서의 형태를 확인하였습니다. 형식은 page_content에는 분할된 텍스트의 본문이 저장되어져 있고, source에는 해당 본문의 원본 파일의 이름이 저장되어져 있습니다.  
2. PDF 페이지 371 -> 문서의 개수가 366개로 줄었다는 것은 이미지는 로드 안 하고 텍스트만 로드한 것입니다.
3. 길이 분포를 시각화하고, 또 최대 길이, 최소 길이, 평균 길이를 계산하여 지나치게 짧거나 지나치게 긴 문서가 없는지 확인했습니다.  
4. 앞의 문서와 뒤의 문서는 머리말과 맺음말인 것을 PDF 파일을 통해 확인하였으므로 실제 출력을 통해 머리말과 맺음말, 목차의 위치를 확인하고 해당 파일들을 제거하였습니다.  

결과적으로 366개의 데이터는 352개의 데이터가 되었습니다.  
이제 형식을 이해하였으며 352개의 금융 문서가 있음을 알았습니다.

## **필요 데이터 전처리**

OpenAI Embedding API를 이용하여 텍스트를 임베딩하고, 코사인 유사도를 통해 유사한 텍스트를 가져오는 실습을 진행한 바 있습니다. 여기서는 OpenAI의 Embedding API를 사용합니다. 일반적으로 OpenAI Embedding API가 sentence_transformer 라이브러리를 이용하는 것보다 성능이 더 뛰어납니다.

Chroma DB는 이 과정들을 기능 별로 이미 구현하여 사용자가 벡터를 좀 더 쉽게 다룰 수 있도록 도와주는 편리한 벡터 응용 도구입니다. 일반적으로 Embedding하여 벡터들 간의 유사도를 구할 때에는 Vector DB라는 것을 사용합니다.  
Chroma.from_documents()를 통해 벡터 도구 객체를 선언합니다. 이때 documents에는 벡터화의 단위가 될 텍스트 리스트를 매개변수로 사용하고, embedding에는 어떤 종류의 임베딩을 사용할 것인지를 기재해줍니다.

크로마 사용 방법: https://python.langchain.com/docs/integrations/vectorstores/chroma  

위 링크에서 아래에 보면 'Use OpenAI Embeddings'라고 해서 OpenAI Embedding을 사용하는 경우의 예시도 나와있습니다.

In [ ]:
# OpenAI 임베딩 설정에서 배치 크기 제한
# chunk_size=100 = 한 번의 API 호출에 100개 문서를 보냄.
embedding = OpenAIEmbeddings(model="text-embedding-3-small", chunk_size=100)
vectordb = Chroma.from_documents(documents=texts, embedding=embedding)

vectordb를 선언하고 나면 그 후에는 '_collection' 다음에 온점을 찍고 다양한 함수들을 사용할 수 있습니다. 예를 들어 count()는 현재 저장된 문서 또는 벡터 개수를 의미합니다.

In [ ]:
# 벡터DB의 개수 확인
vectordb._collection.count()

기본적으로 `_collection.get()`은 현재 vectordb에 저장된 값들을 볼 수 있게 하는 기능을 갖고 있습니다. 어떤 값들을 호출할 수 있는지 확인해봅시다.

In [ ]:
# vectordb._collection.get()

In [ ]:
for key in vectordb._collection.get():
  print(key)

ids, embeddings, metadatas, documents를 호출할 수 있습니다. vectordb에 저장된 기존 문서들을 보고 싶다면 '['documents']'를 통해 불러올 수 있습니다.

In [ ]:
# 문서 로드
documents = vectordb._collection.get()['documents']
print('문서의 개수 :', len(documents))
print('-' * 100)
print('첫번째 문서 출력 :', documents[0])

In [ ]:
# embedding 호출 시도
result = vectordb._collection.get()['embeddings']
print(result)

embedding 벡터의 값은 기본적으로는 제공하지 않기 때문에 embedding 벡터의 값도 확인하고 싶다면 get() 호출 시 내부에 include=['embeddings']를 함께 호출해야 합니다. 그 후 ['embeddings']를 통해 호출할 수 있습니다.

In [ ]:
# embedding vetor만 조회하기
embeddings = vectordb._collection.get(include=['embeddings'])['embeddings']
print('임베딩 벡터의 개수 :', len(embeddings))

In [ ]:
print('첫번째 문서의 임베딩 값 출력 :', embeddings[0])
print('첫번째 문서의 임베딩 값의 길이 :', len(embeddings[0]))

이번에는 metadatas를 호출해봅시다. 참고로 metadatas는 각 문서의 출처를 의미합니다.

In [ ]:
metadatas = vectordb._collection.get()['metadatas']
print('metadatas의 개수 :', len(metadatas))
print('첫번째 문서의 출처 :', metadatas[0])

벡터 도구 객체를 선언하고 나면 as_retriever()를 통해서 입력된 텍스트로부터 유사한 텍스트를 찾아주는 retriever를 선언할 수 있습니다. retriever를 선언 후 invoke()를 통해 입력된 텍스트와 유사한 문서들을 찾아서 반환합니다. 앞서 실전 모델링2에서 실습했던 벡터의 유사도를 구하는 과정을 별도의 추가 구현없이 손쉽게 사용할 수 있습니다. 여기서도 내부적으로 코사인 유사도를 수행하고 있습니다.

In [ ]:
# 유사도가 높은 문서 2개만 추출. k = 2
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

In [ ]:
docs = retriever.invoke("비트코인이 궁금해")
print('유사 문서 개수 :', len(docs))
print('--' * 20)
print('첫번째 유사 문서 :', docs[0])
print('--' * 20)
print('두번째 유사 문서 :', docs[1])

먼저 `ChatPromptTemplate`을 이용해 답변의 기본 틀을 정의합니다.  
이 프롬프트에는 `{context}`와 `{question}` 변수가 포함되어, 검색된 문서 내용과 사용자의 질문이 삽입됩니다.  
프롬프트 내용은 “검색 결과를 바탕으로만 답하라”, “없는 내용은 모른다고 하라”는 명확한 지침을 담고 있습니다.  

`ChatOpenAI`은 GPT-4o 모델을 불러와 실제 답변을 생성하는 역할을 합니다.  
온도(`temperature`)를 0으로 설정하여 일관성 있고 재현 가능한 응답을 유도합니다.  

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# 새 프롬프트 정의
prompt = ChatPromptTemplate.from_template("""
당신은 권현성의 경제금융용어 학습 도우미입니다.
주어진 검색 결과만 사용해 핵심 정의와 쉬운 예시를 차례로 설명하세요.
검색 결과에 근거가 없으면 모른다고 답하고, 황제께 아뢰듯 정중하게 답하세요.

검색 결과:
{context}

질문: {question}

답변:
""")

`query_rag()` 함수는 질의 처리의 핵심 단계입니다.  
먼저 `retriever.invoke(question)`으로 입력된 질문과 유사한 문서들을 벡터 DB에서 검색합니다.  
검색된 문서들의 본문(`page_content`)을 추출해 하나의 문자열 `context`로 합칩니다.  

그다음 `prompt.format_messages()`를 통해 `{question}`과 `{context}`를 실제 값으로 채워 LLM 입력 메시지를 만듭니다.  
이 메시지를 `llm.invoke(messages)`로 전달하면 GPT-4.1가 해당 내용을 바탕으로 답변을 생성합니다.  
모델의 출력은 `response.content` 속성에 저장됩니다.  

마지막으로 함수는 `query`, `result`, `source_documents` 세 항목을 포함한 딕셔너리를 반환합니다.  
`query`에는 원본 질문, `result`에는 모델이 생성한 답변,  
`source_documents`에는 참고된 문서들이 들어 있습니다.  

이 코드는 RAG 구조의 핵심 단계를 단순화하여, 검색 → 프롬프트 구성 → 응답 생성을 명확하게 제어할 수 있게 설계되었습니다.  
따라서 개발자가 모델 응답의 근거를 추적하고, 검색 또는 프롬프트 구성을 쉽게 수정할 수 있습니다.

In [ ]:
# LLM 초기화
llm = ChatOpenAI(model="gpt-4.1", temperature=0)

# 검색 및 응답 함수 (수정 완료 버전)
def query_rag(question):
    # 1. 검색 수행 (invoke로 변경)
    docs = retriever.invoke(question)
    # 2. context 구성
    context = "\n\n".join([doc.page_content for doc in docs])
    # 3. 프롬프트 메시지 구성
    messages = prompt.format_messages(question=question, context=context)
    # 4. LLM 호출
    response = llm.invoke(messages)
    # 5. 결과 반환
    return {
        "query": question,
        "result": response.content,
        "source_documents": docs
    }

이제 query_rag을 통해 사용자의 입력으로부터 서울 청년 정책과 관련된 챗봇의 답변을 얻을 수 있습니다. 임의의 "디커플링이란 무엇인가?"라는 텍스트를 입력하여 query_rag 반환 결과를 확인해봅시다.

In [ ]:
input_text = "기준금리란 무엇이며 생활에 어떤 영향을 주나요?"
chatbot_response = query_rag(input_text)

In [ ]:
chatbot_response

이번에는 경제금융용어 700선 파일로는 알 수 없는 정보를 물어보겠습니다.

In [ ]:
input_text = "인플레이션과 디플레이션의 차이를 알려주세요"
chatbot_response = query_rag(input_text)

In [ ]:
chatbot_response

## Model Inference

In [ ]:
chatbot_response['result']

In [ ]:
def get_chatbot_response(chatbot_response):
    return chatbot_response['result'].strip()

> 실제 회사에서의 업무를 수행할 때는 테스트를 위한 데이터가 아무리 적어도 최소 수십 개는 준비되어져 있어야만 합니다. 그래야만 현재의 모델이 문제가 있다고 판단되었고, 모델을 추후 개선하였을 때 동일한 테스트 데이터에 대해서 얼만큼 개선이 되었는지 정량적으로 평가가 가능하기 때문입니다.

```
1. '너는 뭘하는 챗봇이니?'
- 챗봇을 사용하는 사용자가 반드시 물어볼 수 있는 챗봇의 역할에 대한 질문. 여기서 잘못된 답변이 나가면 사용자가 느끼는 챗봇의 성능이 크게 저하될 것이다.
```

```
2. <최근 가장 핫한 이슈가 되는 대상>에 대해서 궁금해
- 사용자는 금융 용어 챗봇이라면 최근 가장 핫한 이슈가 되는 금융 용어에 대해서 질문할 가능성이 매우 높다. 예를 들어 '비트코인'을 해보자.
```

```
3. <실제 챗봇과 직접적으로, 또 그리고 완전히 연관이 없는 대상>에 대한 문의
- 챗봇은 자신의 도메인을 완전히 벗어난 질문에 대해서 거짓을 말하거나 편향된 답변을 하기보다는 답변을 거부해야할 것이다.
```

이번에는 실제 챗봇과의 답변을 가정하고 사용자의 질문으로부터 챗봇의 답변이 오면 해당 챗봇의 답변으로부터 이어서 사용자가 질문하는 시나리오를 진행해보겠습니다. "너는 뭘하는 챗봇이니?"라는 질문부터 "비트코인에 대해서 궁금하당~"과 같이 사용자가 할 법한 임의의 질문을 입력합니다.

In [ ]:
input_text = "어떤 금융 용어를 설명할 수 있나요?"
llm_response = query_rag(input_text)
result = get_chatbot_response(llm_response)
print(result)

In [ ]:
input_text = "예금자보호제도가 무엇인가요?"
llm_response = query_rag(input_text)
result = get_chatbot_response(llm_response)
print(result)

In [ ]:
input_text = "양적완화가 무엇인가요?"
llm_response = query_rag(input_text)
result = get_chatbot_response(llm_response)
print(result)

In [ ]:
input_text = "오늘 서울 날씨를 알려줘"
llm_response = query_rag(input_text)
result = get_chatbot_response(llm_response)
print(result)

이렇게 다수의 문서로부터 질의 응답을 할 수 있는 챗봇을 구현해보았습니다. 이렇게 구현한 챗봇을 앞으로 사용할 gradio, 카카오톡이나 텔레그램 등을 연동하여 나만의 커스텀 챗봇을 구현할 수 있습니다.

## Model Demo

* Streamlit  
* Gradio

In [ ]:
!pip install gradio

https://www.gradio.app/guides/creating-a-custom-chatbot-with-blocks

위의 gradio 공식 문서 웹 사이트에서 제공하고 있는 Chatbot 코드

```
import gradio as gr
import random
import time

with gr.Blocks() as demo:
    chatbot = gr.Chatbot()
    msg = gr.Textbox()
    clear = gr.ClearButton([msg, chatbot])

    def respond(message, chat_history):
        bot_message = random.choice(["How are you?", "I love you", "I'm very hungry"])
        chat_history.append((message, bot_message))
        time.sleep(2)
        return "", chat_history

    msg.submit(respond, [msg, chatbot], [msg, chatbot])

demo.launch()
```

위 코드를 조금만 수정하여 챗봇을 만들 수 있습니다.

In [ ]:
import gradio as gr

# 인터페이스를 생성.
with gr.Blocks() as demo:
    chatbot = gr.Chatbot(label="권현성의 경제금융용어 챗봇") # 경제금융용어 챗봇 레이블을 좌측 상단에 구성
    msg = gr.Textbox(label="질문해주세요!")  # 하단의 채팅창의 레이블
    clear = gr.ClearButton([msg, chatbot])

    # 챗봇의 답변을 처리하는 함수
    def respond(message, chat_history):
      result = query_rag(message)
      bot_message = result['result']

      # 채팅 기록에 사용자의 메시지와 봇의 응답을 추가.
      chat_history.append((message, bot_message))
      return "", chat_history

    # 사용자의 입력을 제출(submit)하면 respond 함수가 호출.
    msg.submit(respond, [msg, chatbot], [msg, chatbot])

# 인터페이스 실행.
demo.launch()